# Solar Filament Segmentation Challenge 2026 — portable public raw-JPEG inference

**Public inference notebook (13 November 2026).** Run the released H9b model family from the 180 organiser-supplied unlabelled test JPEGs. The pipeline needs no test annotations, private prediction caches, network calls or retraining. It restores the **original FP32 Mask R-CNN checkpoints** from the eight published shards, in addition to four native ResNet34 U-Nets, the frozen TRAIN-only selector and two crop refiners.

Install the pinned repository dependencies (`python -m pip install -r requirements.txt`) and start Jupyter inside this checkout. CPU is the default. The test images default to `data/MAGFiLO_1.0_Kaggle_2026/test/test_images`; use `SOLARFIL_TEST_IMAGES` for another authorised directory. `SOLARFIL_REPLAY_OUT` selects a writable disk with at least 20 GiB free; it defaults to `outputs/h9b_public_cpu`. No author's server path or GPU UUID is required.

For optional GPU execution set `SOLARFIL_DEVICE=cuda`. Multiple visible GPUs require `SOLARFIL_GPU_UUID` (physical UUID or index); existing CUDA visibility restrictions are respected. Existing compute jobs are preserved. Native H3 inference retains the released CUDA autocast policy; the detector uses original FP32 weights without autocast. Refiners run in CPU FP32. CPU and GPU results must not be assumed identical.

**Scientific limitation:** the historical H9b pipeline (submission 58112557, PQ 0.39) defines a fallback to an earlier own-model control prediction when no component survives; use of that branch by the genuine upload is not established here. This notebook uses a documented deterministic semantic fallback instead. Numerical backend differences may also affect masks. This rebuild must not be represented as the exact historical upload, a newly evaluated submission or a measured PQ improvement. Complete independent execution remains to be verified.

Output directories bind input hashes, model weights, sources and backend. Preserve earlier runs and choose a fresh directory for a changed version or backend.

In [ ]:
from __future__ import annotations

import csv
import datetime as dt
import json
import os
from pathlib import Path
import pickle
import subprocess
import sys

import cv2
import numpy as np

_origin = Path.cwd().resolve()
ROOT = next((path for path in (_origin, *_origin.parents)
             if (path / "release/model_metadata/WEIGHTS_SHA256SUMS.txt").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the complete public repository checkout.")
MODELS = ROOT / "release/models"
SOURCES = ROOT / "release/source"
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SOURCES))
sys.path.insert(0, str(SCRIPTS))
from public_replay_runtime import (
    check_gpu_idle, materialize_map, prepare_output, process_environment,
    publish_json, select_device, sha256_file, verify_h3_receipt, notebook_source_sha256,
)

FOLDS = ("F1", "F2", "F3", "F4")
DEVICE, GPU_UUID = select_device(
    os.environ.get("SOLARFIL_DEVICE", "cpu").strip().lower(),
    os.environ.get("SOLARFIL_GPU_UUID") or None,
)
IMAGES = Path(os.environ.get(
    "SOLARFIL_TEST_IMAGES", str(ROOT / "data/MAGFiLO_1.0_Kaggle_2026/test/test_images")
)).expanduser().resolve()
FILES = sorted(IMAGES.glob("*.jpeg"))
if len(FILES) != 180 or len({path.stem for path in FILES}) != 180:
    raise RuntimeError("Expected 180 distinct official competition test JPEGs.")
OUT = prepare_output(Path(os.environ.get(
    "SOLARFIL_REPLAY_OUT", str(ROOT / "outputs" / f"h9b_public_{DEVICE}")
)))
H3 = OUT / "h3"
DETECTORS = OUT / "detectors_fp32"
FP32_MODELS = OUT / "weights_fp32"
OVERLAY = OUT / "pipeline"
CROPS = OUT / "crops"
REFINERS = OUT / "refiners"
CROPS.mkdir(exist_ok=True)
REFINERS.mkdir(exist_ok=True)
cv2.setNumThreads(1)
sha = sha256_file
write_json_once = publish_json

print("Official unlabelled images:", len(FILES), "output:", OUT, "backend:", DEVICE)


## 1. Verify model weights and restore original FP32 detectors

The frozen ten-file release manifest, TRAIN-only selector and eight original-FP32 detector shards are checked before inference. The four original detector files are reassembled locally without downloading code, weights or data. A run contract binds images, sources, weights and backend before any predictions are produced.

In [ ]:
from reassemble_maskrcnn_fp32 import validate_manifest

manifest_file = ROOT / "release/model_metadata/WEIGHTS_SHA256SUMS.txt"
manifest = {}
for line in manifest_file.read_text(encoding="utf8").splitlines():
    expected, basename = line.split()
    if len(expected) != 64 or basename in manifest:
        raise RuntimeError("Malformed or duplicate weight signature.")
    weight = MODELS / basename
    if not weight.is_file() or sha(weight) != expected:
        raise RuntimeError(f"Published checkpoint failed SHA-256: {basename}")
    manifest[basename] = expected
if len(manifest) != 10:
    raise RuntimeError("Expected ten checkpoint files in the frozen public manifest.")
selector_json = ROOT / "release/model_metadata/h9b_nosite_selector.json"
selector_path = ROOT / "release/model_metadata/h9b_nosite_selector.pkl"
selector_meta = json.loads(selector_json.read_text(encoding="utf8"))
if selector_meta["weight_sha256"] != sha(selector_path):
    raise RuntimeError("Frozen TRAIN-only selector integrity failure.")
if not selector_meta["no_competition_test_annotations"]:
    raise RuntimeError("TRAIN-only feature provenance was not certified.")
subprocess.run([sys.executable, str(SCRIPTS / "verify_release_assets.py")],
               check=True, cwd=ROOT)
subprocess.run([
    sys.executable, str(SCRIPTS / "reassemble_maskrcnn_fp32.py"),
    "--output-dir", str(FP32_MODELS),
], check=True, cwd=ROOT)
fp32_manifest = validate_manifest(MODELS / "fp32_shards/manifest.json")
fp32_hashes = {fold: fp32_manifest[fold]["original_sha256"] for fold in FOLDS}
for fold in FOLDS:
    if sha(FP32_MODELS / f"maskrcnn_{fold}_fp32.pt") != fp32_hashes[fold]:
        raise RuntimeError(f"Restored original FP32 checkpoint mismatch: {fold}")

source_paths = sorted(SOURCES.rglob("*.py")) + [
    SCRIPTS / name for name in (
        "replay_h3_from_original_images.py", "replay_maskrcnn_from_original_images.py",
        "public_replay_runtime.py", "official_consensus.py",
        "replay_public_detector_p81.py",
        "p95_nosite_jpeg_crop_audit.py", "reassemble_maskrcnn_fp32.py",
    )
]
run_contract = {
    "format": "PUBLIC_H9B_PORTABLE_RAW_REPLAY_V1",
    "images": {path.name: sha(path) for path in FILES},
    "weights": manifest, "original_fp32_detectors": fp32_hashes,
    "selector_sha256": sha(selector_path),
    "selector_metadata_sha256": sha(selector_json),
    "source_sha256": {str(path.relative_to(ROOT)): sha(path) for path in source_paths},
    "notebook_source_sha256": notebook_source_sha256(ROOT / "notebooks/h9b_clean_raw_jpeg_to_csv.ipynb"),
    "device": DEVICE, "gpu_uuid": GPU_UUID,
    "fallback_policy": "model_only_largest_semantic_component_or_empty",
}
contract_path = OUT / "run_contract.json"
if contract_path.exists():
    if json.loads(contract_path.read_text(encoding="utf8")) != run_contract:
        raise RuntimeError("Output directory belongs to different inputs or code; choose a new directory.")
else:
    write_json_once(contract_path, run_contract)
print("MODEL_SHA256_VERIFIED", len(manifest), "original FP32 detectors", len(fp32_hashes))


## 2. Generate four native-resolution ResNet34 U-Net H3 maps

Each fold runs on the configured CPU or single selected idle GPU. Completed folds are reused only after every source-image and output-map hash is checked. An incomplete fold without its final receipt is preserved for review; select a fresh output directory before recomputing it. CPU and CUDA may produce numerically different maps.

In [ ]:
h3_env = process_environment(DEVICE, GPU_UUID, ROOT)
for fold in FOLDS:
    target = H3 / fold
    receipt = H3 / f"{fold}_h3_replay_receipt.json"
    expected_weight = manifest[f"{fold}_resnet34_s0_fp16.pt"]
    if receipt.exists():
        verify_h3_receipt(receipt, target, FILES, expected_weight, DEVICE)
        print("H3_REUSED_HASH_VERIFIED", fold)
        continue
    if target.exists() and any(target.glob("*.png")):
        raise RuntimeError(
            f"Partial H3 outputs lack a complete receipt: {target}. "
            "Preserve these files for review and choose a new output directory."
        )
    if DEVICE == "cuda":
        check_gpu_idle(GPU_UUID)
    command = [
        sys.executable, "-u", str(SCRIPTS / "replay_h3_from_original_images.py"),
        "--images", str(IMAGES), "--out", str(H3), "--fold", fold, "--device", DEVICE,
    ]
    if DEVICE == "cuda":
        command += ["--gpu-uuid", GPU_UUID]
    subprocess.run(command, cwd=ROOT, env=h3_env, check=True)
    verify_h3_receipt(receipt, target, FILES, expected_weight, DEVICE)
print("FOUR_H3_FOLDS_READY")


## 3. Original FP32 Mask R-CNN proposals

This stage uses byte-verified original FP32 checkpoints, replacing the earlier quantised export. It evaluates two geometric views per image and fold without autocast. CPU is the default and follows the P88 inference settings; optional CUDA may produce numerical differences. Per-image hashes and the execution contract are checked on every restart. No historical proposal cache is required.

In [ ]:
detector_env = process_environment(DEVICE, GPU_UUID, ROOT)
command = [
    sys.executable, "-u", str(SCRIPTS / "replay_maskrcnn_from_original_images.py"),
    "--release", str(ROOT / "release"), "--images", str(IMAGES),
    "--weights", str(FP32_MODELS), "--out", str(DETECTORS), "--device", DEVICE,
]
if DEVICE == "cuda":
    check_gpu_idle(GPU_UUID)
    command += ["--gpu-uuid", GPU_UUID]
# Completed per-image receipts are revalidated; prior runs are never overwritten.
subprocess.run(command, cwd=ROOT, env=detector_env, check=True)
detector_receipt = DETECTORS / "receipt.json"
detector_meta = json.loads(detector_receipt.read_text(encoding="utf8"))
if (detector_meta.get("status") != "PUBLIC_ORIGINAL_FP32_DETECTORS_COMPLETE_NOT_FINAL_CSV"
        or set(detector_meta.get("folds", {})) != set(FOLDS)):
    raise RuntimeError("Incomplete original-FP32 detector stage.")
print("FOUR_ORIGINAL_FP32_MASKRCNN_FOLDS_READY")


## 4. Reconstruct the eight-view consensus and H6 no-site crops

Only stage predictions and the SHA-256-verified TRAIN-trained selector feed the decision rule. The archived historical detector/H3 caches are never inputs. All 180 generated crops are saved separately for recoverability.

In [ ]:
from official_consensus import consensus
from p95_nosite_jpeg_crop_audit import rebuild_one

overlay_dets = OVERLAY / "dets_official.pkl"
OVERLAY.mkdir(parents=True, exist_ok=True)
for fold in FOLDS:
    overlay_maps = OVERLAY / "h3/maps" / f"{fold}_resnet34_s0" / "official"
    overlay_maps.mkdir(parents=True, exist_ok=True)
    for image in FILES:
        materialize_map(H3 / fold / f"{image.stem}.png",
                        overlay_maps / f"{image.stem}.png")

consensus_receipt = OVERLAY / "consensus_receipt.json"
consensus_inputs = {
    "detector_receipt_sha256": sha(detector_receipt),
    "consensus_source_sha256": sha(SCRIPTS / "official_consensus.py"),
}
if not overlay_dets.exists():
    aggregates = {}
    for i, image in enumerate(FILES, 1):
        views = []
        for fold in FOLDS:
            with (DETECTORS / fold / f"{image.stem}.pkl").open("rb") as source:
                pairs = pickle.load(source)
            if len(pairs) != 2:
                raise RuntimeError(f"Expected two detector views: {image.name}")
            views.extend(pairs)
        aggregates[image.name] = consensus(views)
        if i % 30 == 0:
            print("CONSENSUS", i, "/180")
    temporary = overlay_dets.with_suffix(".partial")
    with temporary.open("xb") as sink:
        pickle.dump(aggregates, sink, protocol=pickle.HIGHEST_PROTOCOL)
        sink.flush()
        os.fsync(sink.fileno())
    os.replace(temporary, overlay_dets)
    write_json_once(consensus_receipt, {**consensus_inputs, "output_sha256": sha(overlay_dets)})
else:
    if not consensus_receipt.exists():
        raise RuntimeError("Unattested consensus output requires review.")
    saved = json.loads(consensus_receipt.read_text(encoding="utf8"))
    if saved != {**consensus_inputs, "output_sha256": sha(overlay_dets)}:
        raise RuntimeError("Consensus inputs or output changed.")
    print("CONSENSUS_REUSED_HASH_VERIFIED", sha(overlay_dets))

os.environ["SOLARFIL_ASSET_ROOT"] = str(OVERLAY)
# The historical H9b components() uses this explicit asset root.
import h9_official
if h9_official.O.resolve() != OVERLAY:
    raise RuntimeError("H9b model was imported against a different asset root.")
with selector_path.open("rb") as source:
    selector = pickle.load(source)

crop_receipts = OUT / "crop_receipts"
crop_receipts.mkdir(exist_ok=True)
for i, image in enumerate(FILES, 1):
    target = CROPS / f"{image.stem}.npz"
    evidence = crop_receipts / f"{image.stem}.json"
    if evidence.exists():
        r = json.loads(evidence.read_text(encoding="utf8"))
        if r.get("jpeg_sha256") != sha(image) or r.get("crop_sha256") != sha(target):
            raise RuntimeError(f"Resume crop checksum invalid: {image.name}")
        continue
    if target.exists():
        raise RuntimeError(f"Unattested crop output requires review: {target}")
    result = rebuild_one(image, selector, selector_meta)
    temporary = target.with_suffix(".partial")
    with temporary.open("xb") as sink:
        np.savez_compressed(sink, **result)
        sink.flush()
        os.fsync(sink.fileno())
    os.replace(temporary, target)
    write_json_once(evidence, {"image": image.name,
                              "jpeg_sha256": sha(image),
                              "crop_sha256": sha(target),
                              "instances": int(len(result["comp"]))})
    if i % 20 == 0:
        print("CROPS", i, "/180")
assert len(list(CROPS.glob("*.npz"))) == 180
print("ALL_PUBLIC_CROPS_READY")

## 5. Apply both frozen crop refiners in CPU FP32

The ResNet34 and EfficientNetV2-S refiners use the original four-flip inference scheme. Each completed output is linked to checkpoint, crop and source hashes. This stage launches no CUDA process and does not depend on Kaggle evaluation.

In [ ]:
refiner_files = []
cpu_env = process_environment("cpu", None, ROOT)
crop_hashes = {path.name: sha(path) for path in sorted(CROPS.glob("*.npz"))}
for encoder, checkpoint in (
    ("resnet34", "refiner_ALL.pt"),
    ("efficientnet_v2_s", "refiner_ALL_efficientnet_v2_s.pt"),
):
    result_file = REFINERS / f"{encoder}_probabilities.npz"
    evidence_path = REFINERS / f"{encoder}_receipt.json"
    expected_inputs = {
        "checkpoint_sha256": manifest[checkpoint], "crop_sha256": crop_hashes,
        "inference_source_sha256": sha(SOURCES / "h9_infer.py"), "backend": "cpu_float32",
    }
    if result_file.exists():
        if not evidence_path.exists():
            raise RuntimeError(f"Unattested refiner output requires review: {result_file}")
        previous = json.loads(evidence_path.read_text(encoding="utf8"))
        if previous != {**expected_inputs, "output_sha256": sha(result_file)}:
            raise RuntimeError(f"Refiner inputs or output changed: {encoder}")
    else:
        if evidence_path.exists():
            raise RuntimeError(f"Refiner receipt exists without output: {encoder}")
        temporary = REFINERS / f"{encoder}_probabilities.partial.npz"
        if temporary.exists():
            raise RuntimeError(f"Interrupted refiner output requires review: {temporary}")
        subprocess.run([
            sys.executable, "-u", str(SOURCES / "h9_infer.py"),
            "--weights", str(MODELS / checkpoint), "--encoder", encoder,
            "--crops", str(CROPS), "--out", str(temporary), "--device", "cpu",
        ], cwd=ROOT, env=cpu_env, check=True)
        with np.load(temporary, allow_pickle=False) as probabilities:
            if set(probabilities.files) != {path.stem for path in FILES}:
                raise RuntimeError(f"Refiner output incomplete: {encoder}")
        os.replace(temporary, result_file)
        write_json_once(evidence_path, {**expected_inputs, "output_sha256": sha(result_file)})
    with np.load(result_file, allow_pickle=False) as probabilities:
        if set(probabilities.files) != {path.stem for path in FILES}:
            raise RuntimeError(f"Refiner output incomplete: {encoder}")
    refiner_files.append(result_file)
    print("REFINER_HASH_VERIFIED", encoder, sha(result_file))


## 6. Compose disjoint instances and a COCO-RLE CSV

The H9b mask geometry, stable confidence order and two-refiner probability average are retained. The documented semantic fallback replaces an unavailable historical control prediction; its frequency is recorded. Backend differences remain possible. No test labels are read and no candidate is submitted.

In [ ]:
from filseg.postprocess import rle_counts
import h9_score

compiled_csv = OUT / "submission_public_rebuilt.csv"
compiled_receipt = OUT / "submission_public_rebuilt_receipt.json"
if compiled_receipt.exists():
    prior = json.loads(compiled_receipt.read_text(encoding="utf8"))
    if sha(compiled_csv) != prior["csv_sha256"]:
        raise RuntimeError("Previously completed CSV changed unexpectedly.")
    print("ALREADY_COMPLETED_AND_CHECKED", compiled_csv, prior["csv_sha256"])
else:
    if compiled_csv.exists():
        raise RuntimeError("An unattested output CSV exists; audit it before retrying.")
    pfiles = [np.load(path, allow_pickle=False) for path in refiner_files]
    fallback = []
    allrows = []
    try:
        for i, image in enumerate(FILES, 1):
            stem = image.stem
            with np.load(CROPS / f"{stem}.npz", allow_pickle=False) as archive:
                comp, scores, matrices = archive["comp"], archive["score"], archive["M"]
            prob, unet, mr, labels, areas, peaks = h9_official.components(
                image.name, selector_meta["w"], selector_meta["t"]
            )
            if len(comp):
                mean_probability = np.mean([
                    p[stem].astype(np.float32) for p in pfiles
                ], axis=0).round().astype(np.uint8)
                if mean_probability.shape[0] != len(comp):
                    raise RuntimeError(f"Refiner-crop misalignment: {stem}")
                seeds = [
                    h9_score.refined(labels == int(c) + 1,
                                     mean_probability[j], matrices[j])
                    for j, c in enumerate(comp)
                ]
                result = h9_score.assemble(
                    seeds, np.argsort(-scores, kind="stable")
                )
                masks = [rle_counts(result == v)
                         for v in range(1, int(result.max()) + 1)]
            else:
                # Public-only deterministic replacement for the private earlier
                # baseline. Never misrepresent this as exact historical H9b.
                fallback.append(stem)
                if len(areas):
                    pick = int(np.argmax(areas)) + 1
                    masks = [rle_counts(labels == pick)]
                else:
                    masks = [rle_counts(np.zeros((2048, 2048), dtype=bool))]
            allrows.extend((f"{stem}_{j}", rle)
                           for j, rle in enumerate(masks, 1))
            if i % 20 == 0:
                print("ASSEMBLED", i, "/180")
    finally:
        for p in pfiles:
            p.close()
    if len({ident for ident, _ in allrows}) != len(allrows):
        raise RuntimeError("The output has duplicate instance identifiers.")
    with compiled_csv.open("x", newline="", encoding="utf8") as sink:
        writer = csv.writer(sink, lineterminator="\n")
        writer.writerow(("filament_id", "segmentation_rle"))
        writer.writerows(allrows)
    receipt = {
        "status": "PUBLIC_RAW_IMAGE_REBUILT_NOT_OFFICIAL_SUBMISSION",
        "utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "images": len(FILES), "rows": len(allrows),
        "csv_sha256": sha(compiled_csv),
        "no_site_selector_sha256": sha(selector_path),
        "all_weights_sha256": manifest,
        "original_fp32_detector_sha256": fp32_hashes,
        "run_contract_sha256": sha(contract_path),
        "device": DEVICE, "gpu_uuid": GPU_UUID,
        "fallback_policy": "model_only_largest_semantic_component_or_empty",
        "fallback_images": fallback,
        "test_annotations_used": False,
        "official_kaggle_submission_made": False,
        "exact_historical_upload_parity_asserted": False,
    }
    write_json_once(compiled_receipt, receipt)
    print("PUBLIC_REPLAY_COMPLETED", compiled_csv, receipt["csv_sha256"],
          "rows", receipt["rows"], "fallback", len(fallback))

## 7. Independently validate COCO-RLE masks

The final integrity check decodes each RLE as a 2048 × 2048 COCO mask, verifies that no two submitted instances overlap, checks exact coverage of the 180 official image IDs, and preserves a write-once validation receipt. This is a schema and geometry audit, **not** a PQ measurement.

In [ ]:
from pycocotools import mask as mask_utils

checked = OUT / "geometry_validation_receipt.json"
if checked.exists():
    prev = json.loads(checked.read_text(encoding="utf8"))
    if prev["csv_sha256"] != sha(compiled_csv):
        raise RuntimeError("CSV changed after geometry audit.")
else:
    groups = {}
    with compiled_csv.open(newline="", encoding="utf8") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != ["filament_id", "segmentation_rle"]:
            raise RuntimeError("Wrong Kaggle CSV headers.")
        for entry in reader:
            stem = entry["filament_id"].rsplit("_", 1)[0]
            groups.setdefault(stem, []).append(entry["segmentation_rle"])
    if set(groups) != {p.stem for p in FILES}:
        raise RuntimeError("Submission does not cover exactly 180 test images.")
    checked_masks = 0
    for stem, masks in groups.items():
        occupied = np.zeros((2048, 2048), dtype=bool)
        for encoded in masks:
            binary = mask_utils.decode({
                "size": [2048, 2048], "counts": encoded.encode("ascii")
            }).astype(bool)
            if binary.shape != occupied.shape:
                raise RuntimeError(f"Wrong RLE dimensions for {stem}")
            if np.any(occupied & binary):
                raise RuntimeError(f"Overlapping instances in {stem}")
            occupied |= binary
            checked_masks += 1
    write_json_once(checked, {
        "status": "COCO_RLE_GEOMETRY_VALIDATED_NOT_PQ",
        "csv_sha256": sha(compiled_csv), "images": len(groups),
        "masks": checked_masks, "overlap_failures": 0,
        "hidden_or_test_labels_used": False,
    })
    print("GEOMETRY_VALIDATED", len(groups), checked_masks)

print("REPLAY_OUTPUT", compiled_csv)
print("REPLAY_DIGEST", sha(compiled_csv))

## Submission and scientific integrity

This notebook performs no Kaggle upload. Its output is a public rebuild, not the historical accepted CSV. Public PQ 0.39 (submission 58112557) must not be attributed to it without an official evaluated submission. Original FP32 detector weights improve fidelity to the historical model; they do not prove exact end-to-end masks. H3 backend differences and the semantic fallback remain explicit sources of possible differences.

Validate candidate changes using labelled TRAIN-only calibration and disjoint held-out evaluation. Do not use test labels or optimise the hidden leaderboard. The organiser form requires the actual submitted CSV, four-page report, public repository and receipt. A schema/hash check is not a substitute for full executed scientific validation.